# LeetCode #71 — Simplify Path: chuẩn hóa đường dẫn Unix bằng Stack

**Mức độ:** Medium · **Chủ đề:** String, Stack

Skeleton trên LeetCode:

```python
class Solution:
    def simplifyPath(self, path: str) -> str:
```

Mục tiêu của notebook:

1. Hiểu **ngăn xếp (stack)**: vào sau ra trước (LIFO), với `list.append` / `list.pop`.
2. Tách chuỗi bằng `split('/')` và hiểu vì sao nó sinh ra **chuỗi rỗng**.
3. Mô phỏng việc "đi vào thư mục" (`push`) và "lùi ra thư mục cha" (`pop`).
4. Phân biệt `"."`, `".."` với tên thư mục như `"..."`.

# 1. Bài toán là gì?

Cho `path` là một **đường dẫn tuyệt đối** kiểu Unix (bắt đầu bằng `'/'`). Trả về **đường dẫn chuẩn**.

Quy tắc của hệ thống file Unix (theo đề):

- `'.'` là thư mục **hiện tại**.
- `'..'` là thư mục **cha** (lùi một cấp). Ở thư mục gốc `/` thì `..` vẫn ở `/`.
- Nhiều dấu `'/'` liên tiếp (`'//'`, `'///'`) coi như **một**.
- Mọi chuỗi khác (kể cả `'...'`, `'....'`) là **tên thư mục / file** hợp lệ.

Đường dẫn chuẩn phải:

- Bắt đầu bằng **một** `'/'`.
- Các thư mục cách nhau bằng **đúng một** `'/'`.
- **Không** kết thúc bằng `'/'` (trừ khi là thư mục gốc `"/"`).
- Không chứa `'.'` hay `'..'`.

**Constraints (theo đề):** `1 <= path.length <= 3000`; `path` gồm chữ cái, chữ số, `'.'`, `'/'`, `'_'`; là đường dẫn tuyệt đối hợp lệ.

---

## Ví dụ

| path | kết quả |
|---|---|
| `"/home/"` | `"/home"` |
| `"/home//foo/"` | `"/home/foo"` |
| `"/home/user/Documents/../Pictures"` | `"/home/user/Pictures"` |
| `"/../"` | `"/"` |
| `"/.../a/../b/c/../d/./"` | `"/.../b/d"` |

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Ngăn xếp (stack)
2. Chuỗi (string) trong Python

Cuối mục có phần **2.3 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Ngăn xếp (stack)

### Ngăn xếp là gì?

Ngăn xếp là cấu trúc **vào sau, ra trước** (LIFO — Last In, First Out), giống chồng đĩa: chỉ đặt thêm hoặc lấy ra ở **đỉnh**.

```text
push 1   push 2   push 3   pop → 3   top = 2
                   │ 3 │
          │ 2 │    │ 2 │    │ 2 │
 │ 1 │    │ 1 │    │ 1 │    │ 1 │
 └───┘    └───┘    └───┘    └───┘
```

| Thao tác | Ý nghĩa | Python (`list`) | Độ phức tạp |
|---|---|---|---|
| `push(x)` | đặt `x` lên đỉnh | `st.append(x)` | `O(1)` |
| `pop()` | lấy và bỏ đỉnh | `st.pop()` | `O(1)` |
| `top()` | xem đỉnh | `st[-1]` | `O(1)` |
| rỗng? | | `not st` | `O(1)` |

In [1]:
st = []
for x in [1, 2, 3]:
    st.append(x)
    print("push", x, "->", st)
print("top:", st[-1])
while st:
    print("pop", st.pop(), "->", st)
try:
    st.pop()
except IndexError as e:
    print("pop khi rỗng:", e)

push 1 -> [1]
push 2 -> [1, 2]
push 3 -> [1, 2, 3]
top: 3
pop 3 -> [1, 2]
pop 2 -> [1]
pop 1 -> []
pop khi rỗng: pop from empty list


### Khi nào nghĩ tới ngăn xếp?

Khi thứ được mở **sau cùng** phải được đóng/xử lý **trước tiên**:

- Dấu ngoặc lồng nhau: `( [ { } ] )`.
- Đường dẫn thư mục: `..` hủy thư mục vừa vào.
- Tính biểu thức hậu tố, gọi hàm đệ quy (máy tính dùng *call stack*).
- "Hoàn tác" (undo).

Ví dụ kinh điển — kiểm tra ngoặc hợp lệ: gặp ngoặc mở thì push; gặp ngoặc đóng thì đỉnh phải là ngoặc mở tương ứng.

In [2]:
def ngoac_hop_le(s):
    cap = {")": "(", "]": "[", "}": "{"}
    st = []
    for ch in s:
        if ch in "([{":
            st.append(ch)
        else:
            if not st or st[-1] != cap[ch]:
                return False
            st.pop()
    return not st                        # còn ngoặc mở chưa đóng → sai

for s in ["([]{})", "([)]", "((", "}"]:
    print(f"{s!r:>9}: {ngoac_hop_le(s)}")

 '([]{})': True
   '([)]': False
     '((': False
      '}': False


### Lỗi hay gặp

- `pop()` / `st[-1]` khi ngăn xếp rỗng → `IndexError`. Luôn kiểm tra `if st`.
- Quên kiểm tra ngăn xếp **còn sót** phần tử ở cuối.
- Dùng `list.pop(0)` / `insert(0, x)` (đầu danh sách) — đó là hàng đợi và tốn `O(n)`; ngăn xếp luôn thao tác ở **cuối** list.

## 2.2 Chuỗi (string) trong Python

### Chuỗi là bất biến

`str` **không sửa được** tại chỗ: `s[0] = "x"` gây lỗi. Mọi "sửa" đều tạo chuỗi mới. Vì vậy:

- Ghép nhiều mảnh: gom vào **list** rồi `"".join(list)` (`O(n)`), tránh `s += ...` trong vòng lặp (có thể `O(n²)`).
- Cần sửa từng ký tự: `chars = list(s)`, sửa, rồi `"".join(chars)`.

In [3]:
s_str = "hello"
try:
    s_str[0] = "H"
except TypeError as e:
    print("Sửa trực tiếp:", e)
chars_str = list(s_str); chars_str[0] = "H"
print("qua list:", "".join(chars_str))
manh = []
for i in range(5):
    manh.append(str(i))
print("join:", "-".join(manh))

Sửa trực tiếp: 'str' object does not support item assignment
qua list: Hello
join: 0-1-2-3-4


### Cắt chuỗi (slicing) và các thao tác hay dùng

| Biểu thức | Ý nghĩa |
|---|---|
| `s[i:j]` | ký tự từ `i` tới `j−1` (tạo chuỗi mới, `O(j−i)`) |
| `s[::-1]` | chuỗi đảo ngược |
| `s.split()` | tách theo khoảng trắng (bỏ khoảng trắng thừa) |
| `s.split(",")` | tách theo dấu phẩy (giữ phần rỗng) |
| `s.strip()` | bỏ khoảng trắng hai đầu |
| `s.startswith(p)`, `s.find(p)` | kiểm tra tiền tố, tìm vị trí |
| `c.isdigit()`, `c.isalpha()`, `c.isalnum()`, `c.isspace()` | phân loại ký tự |
| `c.lower()`, `c.upper()` | đổi hoa/thường |

In [4]:
t_str = "  Xin chào   thế giới  "
print(repr(t_str.split()), "|", repr(t_str.strip()), "|", repr("a,,b".split(",")))
print("đảo:", "abcdef"[::-1], "| cắt [1:4]:", "abcdef"[1:4], "| 'abc'.find('c') =", "abc".find("c"))
print([(c, c.isdigit(), c.isalpha()) for c in "a1 "])

['Xin', 'chào', 'thế', 'giới'] | 'Xin chào   thế giới' | ['a', '', 'b']
đảo: fedcba | cắt [1:4]: bcd | 'abc'.find('c') = 2
[('a', False, True), ('1', True, False), (' ', False, False)]


### Ký tự và mã số: `ord` / `chr`

Mỗi ký tự có một mã số (Unicode). `ord("a") = 97`, `chr(97) = "a"`. Mẹo: chữ số `c` → giá trị `ord(c) - ord("0")`; chữ thường `c` → chỉ số 0..25 là `ord(c) - ord("a")`.

In [5]:
print("ord('a') =", ord("a"), "| chr(98) =", chr(98))
print("'7' → 7:", ord("7") - ord("0"), "| 'd' → chỉ số:", ord("d") - ord("a"))
dem26 = [0] * 26
for c in "banana":
    dem26[ord(c) - ord("a")] += 1
print("đếm 26 chữ cái của 'banana' (a, b, n):", dem26[0], dem26[1], dem26[13])

ord('a') = 97 | chr(98) = b
'7' → 7: 7 | 'd' → chỉ số: 3
đếm 26 chữ cái của 'banana' (a, b, n): 3 1 2


### Gom nhóm ký tự liên tiếp

`itertools.groupby` gom các phần tử **liên tiếp bằng nhau** — tiện cho mã hóa độ dài loạt (run-length): `"aaabcc" → 3a 1b 2c`. Tự viết bằng hai con trỏ: `i` ở đầu nhóm, `j` chạy tới khi ký tự khác.

In [6]:
from itertools import groupby
print([(k, len(list(g))) for k, g in groupby("aaabcc")])

def run_length(s):
    out, i = [], 0
    while i < len(s):
        j = i
        while j < len(s) and s[j] == s[i]:
            j += 1
        out.append(f"{j - i}{s[i]}")
        i = j
    return "".join(out)
print("run_length('aaabcc') =", run_length("aaabcc"))

[('a', 3), ('b', 1), ('c', 2)]
run_length('aaabcc') = 3a1b2c


Lỗi hay gặp: `split(" ")` sinh phần rỗng khi có nhiều dấu cách; so sánh chuỗi số `"10" < "9"` là **True** (so từ điển, không phải số); nối chuỗi trong vòng lặp lớn; quên chuỗi rỗng.

## 2.3 Áp dụng vào bài này

Tách đường dẫn bằng `split('/')`, bỏ phần rỗng và `'.'`; mỗi tên thư mục được **push** vào **ngăn xếp**, gặp `'..'` thì **pop** (thư mục mở sau cùng bị đóng trước). Cuối cùng ghép `'/' + '/'.join(stack)`.

# 3. Nền tảng

## 3.1 `split('/')`

Tách chuỗi tại mỗi `'/'`. Hai dấu `'/'` liền nhau, hoặc `'/'` ở đầu/cuối, sinh ra **chuỗi rỗng** `''`:

In [7]:
from typing import List

print("/home//foo/".split("/"))
print("/a/./b/../c".split("/"))
print("/".split("/"))

['', 'home', '', 'foo', '']
['', 'a', '.', 'b', '..', 'c']
['', '']


Các phần tử rỗng `''` ứng với "nhiều dấu `/` liên tiếp" — ta chỉ cần **bỏ qua** chúng. Tương tự, `'.'` cũng bỏ qua (ở yên thư mục hiện tại).

## 3.2 Stack bằng list

- `stack.append(x)`: đẩy `x` lên **đỉnh**.
- `stack.pop()`: lấy ra phần tử ở **đỉnh** (phần tử vào sau cùng).
- `stack[-1]`: xem đỉnh mà không lấy ra.
- `if stack:` kiểm tra rỗng trước khi `pop` (pop trên list rỗng gây `IndexError`).

In [8]:
stack = []
stack.append("home")
stack.append("user")
stack.append("Documents")
print(stack, "đỉnh =", stack[-1])
stack.pop()                 # ".." -> lùi ra
stack.append("Pictures")
print(stack)
print("/" + "/".join(stack))

['home', 'user', 'Documents'] đỉnh = Documents
['home', 'user', 'Pictures']
/home/user/Pictures


Stack chính là **đường đi hiện tại** từ gốc: đi vào thư mục = `push`, `..` = `pop`.

# 4. Lời giải chính

```text
stack = []
for part in path.split('/'):
    if part == '' or part == '.':  bỏ qua
    elif part == '..':             nếu stack không rỗng thì pop
    else:                          push(part)
return '/' + '/'.join(stack)
```

`'/' + '/'.join([])` là `'/'` — thư mục gốc, đúng ý.

In [9]:
def simplify_path(path: str) -> str:
    stack = []
    for part in path.split("/"):
        if part == "" or part == ".":
            continue
        if part == "..":
            if stack:
                stack.pop()
        else:
            stack.append(part)
    return "/" + "/".join(stack)

for p in ["/home/", "/home//foo/", "/home/user/Documents/../Pictures", "/../", "/.../a/../b/c/../d/./"]:
    print(f"{p!r:40} -> {simplify_path(p)!r}")

'/home/'                                 -> '/home'
'/home//foo/'                            -> '/home/foo'
'/home/user/Documents/../Pictures'       -> '/home/user/Pictures'
'/../'                                   -> '/'
'/.../a/../b/c/../d/./'                  -> '/.../b/d'


# 5. Mô phỏng từng bước

In [10]:
def trace_simplify(path: str) -> str:
    stack = []
    print(f"path = {path!r}")
    print(f"split -> {path.split('/')}")
    for part in path.split("/"):
        if part == "":
            action = "rỗng -> bỏ qua"
        elif part == ".":
            action = "'.' -> ở yên"
        elif part == "..":
            if stack:
                action = f"'..' -> pop {stack.pop()!r}"
            else:
                action = "'..' ở gốc -> không làm gì"
        else:
            stack.append(part)
            action = f"push {part!r}"
        print(f"  {part!r:12} {action:28} stack = {stack}")
    result = "/" + "/".join(stack)
    print("Kết quả:", repr(result))
    return result

_ = trace_simplify("/.../a/../b/c/../d/./")

path = '/.../a/../b/c/../d/./'
split -> ['', '...', 'a', '..', 'b', 'c', '..', 'd', '.', '']
  ''           rỗng -> bỏ qua               stack = []
  '...'        push '...'                   stack = ['...']
  'a'          push 'a'                     stack = ['...', 'a']
  '..'         '..' -> pop 'a'              stack = ['...']
  'b'          push 'b'                     stack = ['...', 'b']
  'c'          push 'c'                     stack = ['...', 'b', 'c']
  '..'         '..' -> pop 'c'              stack = ['...', 'b']
  'd'          push 'd'                     stack = ['...', 'b', 'd']
  '.'          '.' -> ở yên                 stack = ['...', 'b', 'd']
  ''           rỗng -> bỏ qua               stack = ['...', 'b', 'd']
Kết quả: '/.../b/d'


Chú ý `'...'` được **push** như một tên thư mục bình thường — chỉ đúng `'.'` và `'..'` là đặc biệt.

# 6. Cách 2: không dùng `split` — tự quét chuỗi

Trong phỏng vấn đôi khi được yêu cầu không dùng `split`. Dùng con trỏ `i` để đọc từng **đoạn** giữa các dấu `'/'`:

In [11]:
def simplify_path_scan(path: str) -> str:
    stack = []
    i, n = 0, len(path)
    while i < n:
        while i < n and path[i] == "/":       # bỏ mọi '/'
            i += 1
        j = i
        while j < n and path[j] != "/":       # đọc một đoạn
            j += 1
        part = path[i:j]
        if part == "..":
            if stack:
                stack.pop()
        elif part and part != ".":
            stack.append(part)
        i = j
    return "/" + "/".join(stack)

print(simplify_path_scan("/home//foo/"), simplify_path_scan("/.../a/../b/c/../d/./"))

/home/foo /.../b/d


# 7. Phiên bản đúng format LeetCode

In [12]:
class Solution:
    def simplifyPath(self, path: str) -> str:
        stack = []
        for part in path.split("/"):
            if part == "" or part == ".":
                continue
            if part == "..":
                if stack:
                    stack.pop()
            else:
                stack.append(part)
        return "/" + "/".join(stack)

In [13]:
import posixpath, random

def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<26} PASS  {actual!r}")


sol = Solution()
check("Ví dụ 1", sol.simplifyPath("/home/"), "/home")
check("Ví dụ 2", sol.simplifyPath("/home//foo/"), "/home/foo")
check("Ví dụ 3", sol.simplifyPath("/home/user/Documents/../Pictures"), "/home/user/Pictures")
check("Ví dụ 4", sol.simplifyPath("/../"), "/")
check("Ví dụ 5", sol.simplifyPath("/.../a/../b/c/../d/./"), "/.../b/d")
check("Chỉ có gốc", sol.simplifyPath("/"), "/")
check("Nhiều '..' vượt gốc", sol.simplifyPath("/a/../../../b"), "/b")
check("Tên có dấu chấm", sol.simplifyPath("/a/..b/.c/"), "/a/..b/.c")
check("Gạch dưới", sol.simplifyPath("/__/x_y/"), "/__/x_y")

random.seed(71)
parts = ["a", "b", "_c", ".", "..", "...", "", "d1"]
for _ in range(5000):
    p = "/" + "/".join(random.choice(parts) for _ in range(random.randint(0, 8)))
    expected = posixpath.normpath(p)
    if expected.startswith("//"):          # normpath giữ "//" ở đầu theo chuẩn POSIX; đề này thì không
        expected = "/" + expected.lstrip("/")
    assert sol.simplifyPath(p) == expected, p
    assert simplify_path_scan(p) == expected, p
print("5000 đường dẫn ngẫu nhiên: khớp posixpath.normpath")

Ví dụ 1                    PASS  '/home'
Ví dụ 2                    PASS  '/home/foo'
Ví dụ 3                    PASS  '/home/user/Pictures'
Ví dụ 4                    PASS  '/'
Ví dụ 5                    PASS  '/.../b/d'
Chỉ có gốc                 PASS  '/'
Nhiều '..' vượt gốc        PASS  '/b'
Tên có dấu chấm            PASS  '/a/..b/.c'
Gạch dưới                  PASS  '/__/x_y'
5000 đường dẫn ngẫu nhiên: khớp posixpath.normpath


Ta dùng `posixpath.normpath` của thư viện chuẩn để **kiểm tra**. Nó gần như làm đúng việc của đề, chỉ khác một chi tiết của chuẩn POSIX: đường dẫn bắt đầu bằng **đúng hai** dấu `'//'` được giữ nguyên — nên khi so sánh ta chuẩn hóa lại phần đầu.

# 8. Vì sao đúng? Độ phức tạp?

**Bất biến:** sau khi xử lý mỗi đoạn, `stack` là danh sách các thư mục trên đường từ gốc tới "thư mục hiện tại" (đã chuẩn hóa).

- Đoạn rỗng / `'.'` không đổi thư mục hiện tại.
- `'..'` lùi về cha (`pop`), trừ khi đang ở gốc.
- Tên khác: đi vào thư mục con (`push`).

Ghép `stack` bằng `'/'` và thêm `'/'` ở đầu cho đúng định dạng chuẩn.

```text
Time:  O(n)   — split và join đều tuyến tính
Space: O(n)   — stack và list các đoạn
```

# 9. Những lỗi dễ mắc

## Lỗi 1: Coi mọi chuỗi bắt đầu bằng `'.'` là đặc biệt

`'...'` là tên thư mục hợp lệ!

In [14]:
def wrong_startswith_dot(path: str) -> str:
    stack = []
    for part in path.split("/"):
        if part.startswith(".."):                  # SAI: '...' cũng bị coi là '..'
            if stack:
                stack.pop()
        elif part == "" or part.startswith("."):
            continue
        else:
            stack.append(part)
    return "/" + "/".join(stack)

print("Sai :", wrong_startswith_dot("/.../a/../b/c/../d/./"), "| Đúng:", sol.simplifyPath("/.../a/../b/c/../d/./"))

Sai : /b/d | Đúng: /.../b/d


## Lỗi 2: `pop` khi stack rỗng

`"/../"` → `stack.pop()` trên list rỗng gây `IndexError`. Phải kiểm tra `if stack`.

In [15]:
def wrong_pop_empty(path: str) -> str:
    stack = []
    for part in path.split("/"):
        if part == "..":
            stack.pop()                            # SAI: chưa kiểm tra rỗng
        elif part and part != ".":
            stack.append(part)
    return "/" + "/".join(stack)

try:
    wrong_pop_empty("/../")
except IndexError as e:
    print("Lỗi:", e)

Lỗi: pop from empty list


## Lỗi 3: Dùng `replace("//", "/")` để gộp dấu `/`

`"///"` sau một lần `replace` vẫn còn `"//"`. Phải lặp, hoặc tốt hơn là dùng `split` (mọi đoạn rỗng tự bị bỏ).

## Lỗi 4: Trả về chuỗi rỗng cho thư mục gốc

Nếu viết `"/".join(stack)` và chỉ thêm `'/'` khi stack khác rỗng, thư mục gốc sẽ thành `""`. Viết `"/" + "/".join(stack)` luôn đúng.

# 10. Mẹo nhận diện pattern

**Stack** khi:

- Có thao tác **"quay lại / hủy bước gần nhất"** (`..`, phím Backspace, Undo).
- Có cấu trúc **lồng nhau** (ngoặc, thẻ HTML, biểu thức).
- Cần xử lý phần tử theo thứ tự **ngược** với lúc thêm vào.

Bài liên quan: **#20 Valid Parentheses**, **#844 Backspace String Compare**, **#1598 Crawler Log Folder**, **#150 Evaluate Reverse Polish Notation**, **#394 Decode String**.

## Template

```python
stack = []
for token in tokens:
    if token là "bỏ qua": continue
    elif token là "quay lại": if stack: stack.pop()
    else: stack.append(token)
```

# 11. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Chuẩn hóa: `"/a//b/./c/../../d/"`, `"/../../x/./y/.."`, `"/./././"`.

## Bài 2 — Điền chỗ trống

In [16]:
def exercise_2_simplify_path(path: str) -> str:
    stack = []
    for part in path.split("/"):
        # TODO: bỏ qua '' và '.', pop với '..' (nếu được), push tên khác
        pass
    return "/" + "/".join(stack)


# assert exercise_2_simplify_path("/home//foo/") == "/home/foo"
# assert exercise_2_simplify_path("/../") == "/"
# assert exercise_2_simplify_path("/.../a/../b/c/../d/./") == "/.../b/d"

## Bài 3 — Biến thể: Backspace String Compare (LeetCode #844)

Ký tự `'#'` là phím xóa (backspace). Hai chuỗi sau khi gõ có bằng nhau không?

```python
backspace_compare("ab#c", "ad#c")  -> True     # cả hai thành "ac"
backspace_compare("a#c", "b")      -> False
backspace_compare("a##c", "#a#c")  -> True     # cả hai thành "c"
```

In [17]:
def exercise_3_backspace_compare(s: str, t: str) -> bool:
    # TODO: dùng stack mô phỏng việc gõ
    pass


# assert exercise_3_backspace_compare("ab#c", "ad#c") is True
# assert exercise_3_backspace_compare("a#c", "b") is False
# assert exercise_3_backspace_compare("a##c", "#a#c") is True

## Bài 4 — Biến thể: đường dẫn tương đối

Cho thư mục hiện tại `cwd` (đã chuẩn) và một đường dẫn `rel` **tương đối** (không bắt đầu bằng `'/'`), hoặc tuyệt đối (bắt đầu bằng `'/'`). Trả về đường dẫn tuyệt đối chuẩn — giống lệnh `cd`.

```python
cd("/home/user", "../admin/./docs") -> "/home/admin/docs"
cd("/home/user", "/etc/../var")     -> "/var"
cd("/", "..")                       -> "/"
```

In [18]:
def exercise_4_cd(cwd: str, rel: str) -> str:
    # TODO
    pass


# assert exercise_4_cd("/home/user", "../admin/./docs") == "/home/admin/docs"
# assert exercise_4_cd("/home/user", "/etc/../var") == "/var"
# assert exercise_4_cd("/", "..") == "/"

## Bài 5 — Mini challenge: Crawler Log Folder (LeetCode #1598)

Danh sách thao tác: `"x/"` vào thư mục con `x`, `"../"` lùi ra (không lùi quá gốc), `"./"` ở yên. Bắt đầu ở gốc; cuối cùng cần **ít nhất bao nhiêu** thao tác `"../"` để về gốc? Yêu cầu `O(1)` bộ nhớ.

```python
min_operations(["d1/", "d2/", "../", "d21/", "./"]) -> 2
min_operations(["d1/", "../", "../", "../"])        -> 0
```

In [19]:
def exercise_5_min_operations(logs: List[str]) -> int:
    # TODO
    pass


# assert exercise_5_min_operations(["d1/", "d2/", "../", "d21/", "./"]) == 2
# assert exercise_5_min_operations(["d1/", "../", "../", "../"]) == 0
# assert exercise_5_min_operations(["d1/", "d2/", "./", "d3/", "../", "d31/"]) == 3

# 12. Gợi ý

**Bài 1**
- Mức 1: dùng `trace_simplify` sau khi tự làm.

**Bài 3**
- Mức 1: viết `build(s)`: với mỗi ký tự, `'#'` → pop (nếu có), ngược lại push; trả `"".join(stack)`.

**Bài 4**
- Mức 1: nếu `rel` bắt đầu bằng `'/'` thì chỉ chuẩn hóa `rel`.
- Mức 2: ngược lại chuẩn hóa `cwd + "/" + rel`.

**Bài 5**
- Mức 1: chỉ cần **độ sâu** hiện tại (độ dài stack), không cần lưu tên.

# 13. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [20]:
for p in ["/a//b/./c/../../d/", "/../../x/./y/..", "/./././"]:
    print(f"{p!r:24} -> {sol.simplifyPath(p)!r}")

'/a//b/./c/../../d/'     -> '/a/d'
'/../../x/./y/..'        -> '/x'
'/./././'                -> '/'


## Đáp án Bài 2

In [21]:
def answer_2_simplify_path(path: str) -> str:
    stack = []
    for part in path.split("/"):
        if part == "" or part == ".":
            continue
        if part == "..":
            if stack:
                stack.pop()
        else:
            stack.append(part)
    return "/" + "/".join(stack)

assert answer_2_simplify_path("/home//foo/") == "/home/foo"
assert answer_2_simplify_path("/../") == "/"
assert answer_2_simplify_path("/.../a/../b/c/../d/./") == "/.../b/d"
print("Bài 2 PASS")

Bài 2 PASS


## Đáp án Bài 3

In [22]:
def answer_3_backspace_compare(s: str, t: str) -> bool:
    def build(text):
        stack = []
        for ch in text:
            if ch == "#":
                if stack:
                    stack.pop()
            else:
                stack.append(ch)
        return "".join(stack)
    return build(s) == build(t)

assert answer_3_backspace_compare("ab#c", "ad#c") is True
assert answer_3_backspace_compare("a#c", "b") is False
assert answer_3_backspace_compare("a##c", "#a#c") is True
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [23]:
def answer_4_cd(cwd: str, rel: str) -> str:
    if rel.startswith("/"):
        return sol.simplifyPath(rel)
    return sol.simplifyPath(cwd + "/" + rel)

assert answer_4_cd("/home/user", "../admin/./docs") == "/home/admin/docs"
assert answer_4_cd("/home/user", "/etc/../var") == "/var"
assert answer_4_cd("/", "..") == "/"
print("Bài 4 PASS")

Bài 4 PASS


Mẹo: nối `cwd + "/" + rel` có thể sinh `"//"` (khi `cwd = "/"`) — không sao, vì hàm chuẩn hóa đã bỏ qua đoạn rỗng.

## Đáp án Bài 5

In [24]:
def answer_5_min_operations(logs: List[str]) -> int:
    depth = 0
    for op in logs:
        if op == "../":
            depth = max(0, depth - 1)
        elif op != "./":
            depth += 1
    return depth

assert answer_5_min_operations(["d1/", "d2/", "../", "d21/", "./"]) == 2
assert answer_5_min_operations(["d1/", "../", "../", "../"]) == 0
assert answer_5_min_operations(["d1/", "d2/", "./", "d3/", "../", "d31/"]) == 3
print("Bài 5 PASS")

Bài 5 PASS


Khi chỉ cần **kích thước** của stack, thay stack bằng một bộ đếm — bộ nhớ `O(1)`.

# 14. Tổng kết

> Tách theo `'/'`; bỏ qua đoạn rỗng và `'.'`; `'..'` thì `pop` (nếu không ở gốc); tên khác thì `push`. Kết quả là `'/' + '/'.join(stack)`.

---

## Checklist trước khi nộp

- [ ] Bỏ qua đoạn rỗng (do `'//'` hoặc `'/'` ở cuối)?
- [ ] Chỉ đúng `'.'` và `'..'` là đặc biệt (`'...'` là tên)?
- [ ] Kiểm tra stack rỗng trước khi `pop`?
- [ ] Thư mục gốc trả `"/"`?